# 已有模型的风险归因与预测头诊断

只读QLib recorder b94059c2e0754b0ab102e4f62534eefb。固定HAC20、已有20/60日相关块与2000次重采样；不改模型、不选新参数。静态对照只用各年的先前成熟训练窗口确定平均仓位，禁止使用完整评估期拟合暴露。

In [1]:
import os, json, hashlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import sys
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
import numpy as np, pandas as pd, qlib, torch
from scipy.stats import norm, spearmanr
from qlib.workflow import R
from src.data.integrated_allocation import (PRIMARY_FEATURES, PRIMARY_LABELS, PRIMARY_METADATA, FORECAST_LABELS,
    RETURN_LABELS,build_native_dataset,prepare_mature_rows,frame_sha256)
from src.objective.relative_wealth import stationary_block_bootstrap_indices
from src.backtest.position_accounting import settle_target_session
from src.workflow.stage1_economic_action import fee_affordable_target
from src.model.integrated_allocation import policy_target

summary=json.loads((root/'research/stable_excess_integration_20261002/summary.json').read_text())
assert summary['recorder_id']=='b94059c2e0754b0ab102e4f62534eefb'
qlib.init(provider_uri=str(root/summary['provider_uri']),region='us',kernels=1)
with R.uri_context(summary['recorder_uri']):
    recorder=R.get_recorder(experiment_name=summary['experiment'],recorder_id=summary['recorder_id'])
    raw=recorder.load_object('inputs/primary_frame.pkl')
    forecast=recorder.load_object('inputs/forecast_frame.pkl')
    prices=recorder.load_object('inputs/execution_prices.pkl')
    audit=recorder.load_object('inputs/source_audit.pkl')
    paths=recorder.load_object('analysis/portfolio_replays.pkl')
    models=recorder.load_object('models/development.pkl')
assert frame_sha256(raw)==audit['primary_frame_sha256']
assert frame_sha256(prices)==audit['execution_prices_sha256']
for field in PRIMARY_FEATURES[:-3]:
    np.testing.assert_array_equal(raw[field],forecast[field])
print('LOADED',len(raw),len(paths),flush=True)

def hac_regression(y,x,lags=20):
    y,x=np.asarray(y,float),np.asarray(x,float)
    X=np.c_[np.ones(len(x)),x]
    coef=np.linalg.lstsq(X,y,rcond=None)[0]
    residual=y-X@coef
    bread=np.linalg.inv(X.T@X)
    score=X*residual[:,None]
    meat=score.T@score
    for lag in range(1,min(lags,len(x)-1)+1):
        cross=score[lag:].T@score[:-lag]
        meat+=(1-lag/(lags+1))*(cross+cross.T)
    covariance=bread@meat@bread*len(x)/(len(x)-2)
    se=np.sqrt(np.maximum(np.diag(covariance),0))
    z=coef/se
    return {'n':len(x),'intercept':coef[0],'slope':coef[1], 'intercept_se_hac':se[0],'slope_se_hac':se[1],
        'intercept_z_hac':z[0],'slope_z_hac':z[1],'intercept_p_normal_two_sided':2*norm.sf(abs(z[0])),
        'intercept_ci95_low':coef[0]-norm.ppf(.975)*se[0], 'intercept_ci95_high':coef[0]+norm.ppf(.975)*se[0],
        'r_squared':1-float(residual@residual)/float(((y-y.mean())**2).sum()),'lags':lags,'kernel':'Bartlett','small_sample_correction':True}

attribution=[]
development=paths[paths.segment.eq('development') & paths.variant.isin(['base','pit_auxiliary','cash_vol_forecasts'])]
for (variant,seed,cost),p in development.groupby(['variant','seed','cost_bps']):
    p=p.sort_values('signal_date')
    benchmark=raw.loc[p.signal_date,'soxx_return'].to_numpy(float)
    result=hac_regression(p.strategy_return,benchmark)
    result.update(variant=variant,seed=str(seed),cost_bps=float(cost),
        annualized_mean_excess_pct=25200*(float(p.strategy_return.mean())-float(benchmark.mean())),
        beta_component_annualized_pct=25200*(result['slope']-1)*float(benchmark.mean()),
        annualized_arithmetic_alpha_pct=25200*result['intercept'],
        annualized_alpha_ci95_low_pct=25200*result['intercept_ci95_low'],
        annualized_alpha_ci95_high_pct=25200*result['intercept_ci95_high'],
        strategy_annualized_vol_pct=100*np.std(p.strategy_return,ddof=1)*np.sqrt(252),
        benchmark_annualized_vol_pct=100*np.std(benchmark,ddof=1)*np.sqrt(252),
        mean_target_effective_beta=float((3*p.soxl_weight+p.soxx_weight).mean()))
    attribution.append(result)
print('ATTRIBUTION',pd.DataFrame(attribution).query("seed=='ensemble'")[['variant','cost_bps','slope','annualized_arithmetic_alpha_pct','annualized_alpha_ci95_low_pct','annualized_alpha_ci95_high_pct','r_squared']].to_json(orient='records'),flush=True)

# All OOF rows were forecast using the preceding year's matured training set.
# Comparators are fixed before inspecting losses: prior training target mean,
# and for volatility a 20-observed-open-return sample standard deviation.
dataset=build_native_dataset(raw,feature_columns=PRIMARY_FEATURES,label_columns=PRIMARY_LABELS,
    metadata_columns=PRIMARY_METADATA,as_of=summary['source_as_of'])
open_vol20=prices[('open','SOXX')].pct_change(fill_method=None).rolling(20,min_periods=20).std(ddof=1)
head_rows=[]; head_training=[]
for year in range(2016,2023):
    train,a=prepare_mature_rows(dataset,decision_at=pd.Timestamp(f'{year}-01-01',tz='America/New_York'),
        label_columns=FORECAST_LABELS,availability_column='forecast_available_at')
    test=forecast[forecast.index.year==year].copy()
    assert test.forecast_available.eq(1).all()
    head_training.append(dict(year=year,**a))
    for name,predcol in [('cash_advantage_1d','cash_advantage_prediction'),('future_vol_5d','future_vol_prediction')]:
        for day,row in test.iterrows():
            if not np.isfinite(row[name]):continue
            head_rows.append({'signal_date':day,'year':year,'target':name,'y':float(row[name]),'prediction':float(row[predcol]),
                'past_mean':float(train[name].mean()),'past_open_vol20':float(open_vol20.loc[day]) if name=='future_vol_5d' else np.nan})
head_frame=pd.DataFrame(head_rows)
head_metrics=[]
for (target,year),g in head_frame.groupby(['target','year']):
    base=g.past_mean.to_numpy();y=g.y.to_numpy();p=g.prediction.to_numpy();mse=float(np.mean((y-p)**2))
    calibration=hac_regression(y,p)
    row={'target':target,'year':int(year),'n':len(g),'mse':mse,'baseline_mean_mse':float(np.mean((y-base)**2)),
        'mse_skill_vs_past_mean':1-mse/float(np.mean((y-base)**2)),'pearson':float(np.corrcoef(y,p)[0,1]),
        'spearman':float(spearmanr(y,p).statistic),'prediction_mean':float(p.mean()),'target_mean':float(y.mean()),
        'prediction_std':float(p.std(ddof=1)),'target_std':float(y.std(ddof=1)),
        'calibration_intercept':calibration['intercept'],'calibration_slope':calibration['slope'],
        'calibration_slope_se_hac20':calibration['slope_se_hac'],
        'negative_prediction_fraction':float(np.mean(p<0))}
    if target=='future_vol_5d':row['mse_skill_vs_past_open_vol20']=1-mse/float(np.mean((y-g.past_open_vol20.to_numpy())**2))
    head_metrics.append(row)
head_totals=[];head_bootstrap=[]
for target,g in head_frame.groupby('target'):
    g=g.sort_values('signal_date');y=g.y.to_numpy();p=g.prediction.to_numpy();model_loss=(y-p)**2
    calibration=hac_regression(y,p)
    total={'target':target,'n':len(g),'pearson':float(np.corrcoef(y,p)[0,1]),'spearman':float(spearmanr(y,p).statistic),
        'calibration':calibration,'positive_years_vs_mean':sum(x['mse_skill_vs_past_mean']>0 for x in head_metrics if x['target']==target),
        'negative_prediction_fraction':float(np.mean(p<0)), 'prediction_mean':float(p.mean()),'target_mean':float(y.mean())}
    baselines=['past_mean']+(['past_open_vol20'] if target=='future_vol_5d' else [])
    for baseline in baselines:
        loss=(y-g[baseline].to_numpy())**2;delta=loss-model_loss
        total['mse_skill_vs_'+baseline]=1-float(model_loss.mean()/loss.mean())
        for block in (20,60):
            indices=stationary_block_bootstrap_indices(observation_count=len(g),path_length=len(g),path_count=2000,
                expected_block_length=block,seed=20261002)
            improvements=delta[indices].mean(axis=1)
            head_bootstrap.append({'target':target,'baseline':baseline,'mean_block':block,'resamples':2000,
                'mean_squared_loss_improvement':float(delta.mean()),'ci95_low':float(np.quantile(improvements,.025)),
                'ci95_high':float(np.quantile(improvements,.975)), 'fraction_positive_not_pvalue':float(np.mean(improvements>0)),
                'relative_mean_improvement':float(delta.mean()/loss.mean()),
                'relative_ci95_low':float(np.quantile(improvements,.025)/loss.mean()),
                'relative_ci95_high':float(np.quantile(improvements,.975)/loss.mean())})
    head_totals.append(total)
print('HEAD_TOTALS',json.dumps(head_totals,default=str),flush=True)

# No model refitting: extract the exact retained two-layer policy for faster
# prior-training exposure calibration, verified against its public Torch API.
def pack_model(model):
    n=model.allocator.network
    return [model.feature_mean.detach().numpy(),model.feature_scale.detach().numpy(),
        n[0].weight.detach().numpy(),n[0].bias.detach().numpy(),n[2].weight.detach().numpy(),n[2].bias.detach().numpy()]
def numpy_target(packs,x,w):
    result=[]
    for mean,scale,w1,b1,w2,b2 in packs:
        h=np.tanh(w1@np.r_[(x-mean)/scale,w]+b1);z=w2@h+b2;v=np.exp(z-z.max());result.append(v/v.sum())
    target=np.mean(result,axis=0);return target/target.sum()

def prior_mean_target(annual_models,train):
    packs=[pack_model(m) for m in annual_models]
    features=train.loc[:,list(PRIMARY_FEATURES)].to_numpy(float);returns=train.loc[:,list(RETURN_LABELS)].to_numpy(float)
    current=np.array([0.,0.,1.]);targets=[];maximum_parity_error=0.
    # Paying fees to achieve specified net weights scales all asset values by
    # the same net-NAV factor; therefore normalized subsequent weights depend
    # on net target and market return, not that scalar. This calibration uses
    # that identity only; all evaluated control P&L below uses shared settlement.
    for i,(x,r) in enumerate(zip(features,returns)):
        target=numpy_target(packs,x,current)
        if i in (0,len(features)//2,len(features)-1):
            error=float(np.max(np.abs(target-policy_target(annual_models,x,current))));maximum_parity_error=max(maximum_parity_error,error)
            assert error<1e-12,error
        targets.append(target)
        values=target*np.r_[1+r,1.]
        current=values/values.sum()
    return np.mean(targets,axis=0),maximum_parity_error

control_weights=[];static_results=[];static_paths={}
for family in ['base','pit_auxiliary','cash_vol_forecasts']:
    source=forecast if family=='cash_vol_forecasts' else raw
    ds=build_native_dataset(source,feature_columns=PRIMARY_FEATURES,label_columns=PRIMARY_LABELS,
        metadata_columns=PRIMARY_METADATA,as_of=summary['source_as_of'])
    targets={}
    for year in range(2016,2023):
        train,a=prepare_mature_rows(ds,decision_at=pd.Timestamp(f'{year}-01-01',tz='America/New_York'),
            label_columns=RETURN_LABELS,availability_column='return_available_at')
        annual_models=[models[family][seed][year] for seed in (17,41,71)]
        target,error=prior_mean_target(annual_models,train)
        targets[year]=target
        control_weights.append({'family':family,'year':year,'weights':target.tolist(),'effective_beta':float(3*target[0]+target[1]),
            'last_training_signal':a['last_signal_date'],'last_label_available_at':a['label_available_at_max'],
            'maximum_numpy_torch_target_error':error,'method':'freeze preceding mature-training average target; not fit on evaluation returns'})
    market=source[source.index.year.isin(range(2016,2023)) & source.soxl_return.notna() & source.soxx_return.notna()]
    for cost in (10.,25.):
        current=np.array([0.,0.,1.]);previous_year=None;rows=[]
        for day,r in market.iterrows():
            target=targets[day.year] if day.year!=previous_year else None
            executable=fee_affordable_target(target,pretrade_weights=current,cost_bps=cost) if target is not None else None
            settled=settle_target_session(current,open_target_weights=executable,asset_returns=[r.soxl_return,r.soxx_return,0.],cost_bps=cost)
            rows.append({'signal_date':day,'strategy_return':settled['net_return'],'turnover':settled['turnover']})
            current=settled['ending_weights'];previous_year=day.year
        control=pd.DataFrame(rows);static_paths[(family,cost)]=control
        candidate=development[development.variant.eq(family)&development.seed.eq('ensemble')&development.cost_bps.eq(cost)].sort_values('signal_date')
        assert candidate.signal_date.tolist()==control.signal_date.tolist()
        def metrics(r):
            wealth=np.r_[1.,np.cumprod(1+np.asarray(r))]
            return float(wealth[-1]),float((wealth/np.maximum.accumulate(wealth)-1).min())
        cw,cd=metrics(candidate.strategy_return);bw,bd=metrics(control.strategy_return)
        factor=raw.loc[control.signal_date,'soxx_return'].to_numpy(float)
        beta=hac_regression(control.strategy_return,factor)
        difference=np.log1p(candidate.strategy_return.to_numpy())-np.log1p(control.strategy_return.to_numpy())
        static_results.append({'family':family,'cost_bps':cost,'candidate_return_pct':100*(cw-1),'control_return_pct':100*(bw-1),
            'candidate_vs_prior_fit_control_wealth':cw/bw,'candidate_maxdd_pct':100*cd,'control_maxdd_pct':100*bd,
            'control_turnover':float(control.turnover.sum()),'control_realized_beta':beta['slope'],
            'candidate_minus_control_annualized_log_growth':float(252*difference.mean()),
            'control_definition':'annual initial mix fit only from prior mature training model trajectory; unchanged shares within year; actual net fees'})
    print('STATIC_DONE',family,flush=True)

result={'source_recorder_id':summary['recorder_id'],'source_as_of':summary['source_as_of'],
    'source_primary_sha256':audit['primary_frame_sha256'],'diagnostic_created_at':pd.Timestamp.now(tz='UTC').isoformat(),
    'attribution':attribution,'head_annual_metrics':head_metrics,'head_totals':head_totals,'head_training_audit':head_training,
    'paired_head_loss_bootstrap':head_bootstrap,'prior_training_static_control_weights':control_weights,'static_comparisons':static_results,
    'limitations':['All data and trained families are previously reviewed; diagnostic attribution is not clean forward alpha.',
      'HAC20 normal confidence intervals are descriptive and do not correct past feature/model search.',
      'Paired block-bootstrap loss intervals are conditional diagnostic95percent intervals, not new promotion gates.',
      'Prior-training average-allocation control is causal but not guaranteed to match out-of-sample beta or volatility exactly.',
      'Fees10/25bp and cash return0 follow source evidence; historical session-open prices are not verified live fills.']}
Path('/tmp/soxl_phase1_attribution_metrics.json').write_text(json.dumps(result,ensure_ascii=False,indent=2,default=str)+'\n')
print('STATIC_COMPARISONS',json.dumps(static_results,default=str),flush=True)
print('RESULT /tmp/soxl_phase1_attribution_metrics.json',flush=True)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[61090:MainThread](2026-10-02 18:18:01,557) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61090:MainThread](2026-10-02 18:18:01,559) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61090:MainThread](2026-10-02 18:18:01,559) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stable_excess_integration_20261002')}


LOADED 3956 51416


...

[61090:MainThread](2026-10-02 18:18:02,867) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[61090:MainThread](2026-10-02 18:18:02,867) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


STATIC_DONE cash_vol_forecasts


STATIC_COMPARISONS [{"family": "base", "cost_bps": 10.0, "candidate_return_pct": 989.4608588818089, "control_return_pct": 468.5394090016287, "candidate_vs_prior_fit_control_wealth": 1.9162451039144903, "candidate_maxdd_pct": -73.16397144597559, "control_maxdd_pct": -55.93521001191011, "control_turnover": 1.5286701023234035, "control_realized_beta": 1.4265609073971104, "candidate_minus_control_annualized_log_growth": 0.09301511591020258, "control_definition": "annual initial mix fit only from prior mature training model trajectory; unchanged shares within year; actual net fees"}, {"family": "base", "cost_bps": 25.0, "candidate_return_pct": 490.4728399632191, "control_return_pct": 467.23657686555845, "candidate_vs_prior_fit_control_wealth": 1.0409639717277397, "candidate_maxdd_pct": -75.39888034900667, "control_maxdd_pct": -55.95401667005682, "control_turnover": 1.5286331037899559, "control_realized_beta": 1.426620121036745, "candidate_minus_control_annualized_log_growth": 0.005741821393

RESULT /tmp/soxl_phase1_attribution_metrics.json


## 中文结论

读取冻结QLib recorder `b94059c2e0754b0ab102e4f62534eefb` 的2016–2022信号年、1762个持有区间，完成三组模型、三个种子及组合在10/25bp下的风险归因；没有重训、筛参数或更改现有模型。

- 三组组合实际回归beta约1.80–2.16。10bp下HAC20的年化算术alpha点估计为6.17%–7.75%，但95%区间全部跨零；25bp下点估计均转负。高终值没有证明稳定的风险调整后超额。
- 现金优势头相对过去均值的总体MSE改善仅0.158%，7年中4年改善，20/60日相关块区间均跨零。波动率头相关系数0.627，相对过去均值改善42.22%；相对可直接观察的过去20日开盘收益波动率改善12.40%，其中20日块区间仍跨零，增量证据较弱。
- 每年只用先前成熟训练窗口确定并冻结平均仓位的静态对照表明：25bp下base/PIT/预测头组合相对其各自对照的终值比分别1.041/0.947/1.163。预测头组合回撤-84.39%，对照-57.76%；对照实际beta1.48低于组合2.16，不能把全部收益差称为择时alpha。
- 所有这些窗口已经被研究过，HAC和bootstrap区间属于条件诊断，未覆盖长期试验选择偏差；此次不产生新的晋升资格。

下一步保留有可学性证据的波动信息，先验证它是否改善统一仓位模型的风险与成本；不将弱现金优势头或高杠杆收益误认为已获稳定超额。具体逐年及全部种子数据见 `/tmp/soxl_phase1_attribution_metrics.json`。
